# Hotel Bookings: Delta Lake Pipeline with Data Quality Checks

Bronze -> Silver -> Gold pipeline over 119,390 hotel bookings.

- **Bronze**: raw CSV loaded as-is into a Delta table, with ingest metadata.
- **Silver**: typed and cleaned. Every row is checked against data quality rules;
  rows that fail go to a quarantine table instead of being silently dropped.
- **Gold**: cancellation-rate metrics by hotel, month, and market segment.
- **Quality log**: every run writes its check results to a Delta table, so
  data quality can be tracked over time.

**Setup (Databricks Free Edition):** run the first two cells, then upload
`data.csv` into the `raw` volume they create
(Catalog > workspace > hotel > raw > Upload to this volume). Then run all.

In [0]:
import os
from datetime import datetime, timezone

# On Databricks, tables live in Unity Catalog (catalog.schema.table).
# Locally (for testing), they live in the default Spark catalog.
ON_DATABRICKS = "DATABRICKS_RUNTIME_VERSION" in os.environ

CATALOG = "workspace" if ON_DATABRICKS else "spark_catalog"
SCHEMA = "hotel"
RAW_PATH = (
    f"/Volumes/{CATALOG}/{SCHEMA}/raw/data.csv"
    if ON_DATABRICKS
    else os.environ.get("HOTEL_CSV", "data.csv")
)


def t(name):
    """Fully qualified table name."""
    return f"{CATALOG}.{SCHEMA}.{name}"


spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
if ON_DATABRICKS:
    spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.raw")

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
print(f"Run {RUN_ID}: reading {RAW_PATH}, writing to {CATALOG}.{SCHEMA}")

Run 20261026T223920Z: reading /Volumes/workspace/hotel/raw/data.csv, writing to workspace.hotel


## Bronze: raw ingest
Read every column as a string so nothing is lost or guessed at load time.

In [0]:
from pyspark.sql import functions as F

bronze = (
    spark.read.option("header", True)
    .option("inferSchema", False)
    .csv(RAW_PATH)
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_run_id", F.lit(RUN_ID))
)
if ON_DATABRICKS:
    bronze = bronze.withColumn("_source_file", F.col("_metadata.file_path"))
else:
    bronze = bronze.withColumn("_source_file", F.lit(RAW_PATH))

bronze.write.format("delta").mode("overwrite").option(
    "overwriteSchema", True
).saveAsTable(t("bookings_bronze"))

bronze_count = spark.table(t("bookings_bronze")).count()
print(f"Bronze rows: {bronze_count:,}")

Bronze rows: 119,390


## Silver: clean, type, and check
The source marks missing values as the strings `NA` and `NULL`.

In [0]:
raw = spark.table(t("bookings_bronze"))
data_cols = [c for c in raw.columns if not c.startswith("_")]

# 1. Turn "NA" / "NULL" / "" into real nulls, and count them per column.
cleaned = raw.select(
    *[
        F.when(F.trim(F.col(c)).isin("NA", "NULL", ""), None)
        .otherwise(F.col(c))
        .alias(c)
        for c in data_cols
    ]
)

null_counts = cleaned.select(
    *[F.sum(F.col(c).isNull().cast("int")).alias(c) for c in data_cols]
).first().asDict()
cols_with_nulls = {c: n for c, n in null_counts.items() if n}
print("Columns with missing values:", cols_with_nulls)

# 2. Cast to proper types.
int_cols = [
    "is_canceled", "lead_time", "arrival_date_year", "arrival_date_week_number",
    "arrival_date_day_of_month", "stays_in_weekend_nights", "stays_in_week_nights",
    "adults", "children", "babies", "is_repeated_guest", "previous_cancellations",
    "previous_bookings_not_canceled", "booking_changes", "agent", "company",
    "days_in_waiting_list", "required_car_parking_spaces", "total_of_special_requests",
]
typed = cleaned
for c in int_cols:
    typed = typed.withColumn(c, F.col(c).cast("double").cast("int"))
typed = typed.withColumn("adr", F.col("adr").cast("double"))

# 3. Fill missing values. A missing agent or company means "booked without one".
typed = typed.fillna({"children": 0, "agent": 0, "company": 0, "country": "Unknown"})

# 4. Drop leaky columns: reservation_status is only known after the stay,
#    so it would leak the answer into any cancellation model.
LEAKY = ["reservation_status", "reservation_status_date"]
typed = typed.drop(*LEAKY)

# 5. Add a real arrival date.
typed = typed.withColumn(
    "arrival_date",
    F.to_date(
        F.concat_ws(
            "-",
            F.col("arrival_date_year"),
            F.col("arrival_date_month"),
            F.col("arrival_date_day_of_month"),
        ),
        "yyyy-MMMM-d",
    ),
)

Columns with missing values: {'children': 4, 'country': 488, 'agent': 16340, 'company': 112593}


### Data quality rules
Each rule is a condition a good row must satisfy. Rows failing any rule
are quarantined with the list of rules they broke.

In [0]:
RULES = {
    "valid_hotel": F.col("hotel").isin("Resort Hotel", "City Hotel"),
    "valid_cancel_flag": F.col("is_canceled").isin(0, 1),
    "has_guests": (F.col("adults") + F.col("children") + F.col("babies")) > 0,
    "non_negative_price": F.col("adr") >= 0,
    "non_negative_lead_time": F.col("lead_time") >= 0,
    "has_stay_nights_or_canceled": (
        (F.col("stays_in_weekend_nights") + F.col("stays_in_week_nights")) > 0
    )
    | (F.col("is_canceled") == 1)
    | (F.col("adults") + F.col("children") + F.col("babies") == 0),
    "valid_arrival_date": F.col("arrival_date").isNotNull(),
}

failed_rules = F.array_compact(
    F.array(
        *[F.when(~F.coalesce(cond, F.lit(False)), F.lit(name)) for name, cond in RULES.items()]
    )
)
checked = typed.withColumn("_failed_rules", failed_rules)

silver = checked.filter(F.size("_failed_rules") == 0).drop("_failed_rules")
quarantine = checked.filter(F.size("_failed_rules") > 0).withColumn(
    "_run_id", F.lit(RUN_ID)
)

silver.write.format("delta").mode("overwrite").option(
    "overwriteSchema", True
).saveAsTable(t("bookings_silver"))
quarantine.write.format("delta").mode("overwrite").option(
    "overwriteSchema", True
).saveAsTable(t("bookings_quarantine"))

# Enforce the key rules at the table level too, so future writes can't break them.
for name, sql in [
    ("valid_cancel_flag", "is_canceled IN (0, 1)"),
    ("non_negative_price", "adr >= 0"),
]:
    spark.sql(f"ALTER TABLE {t('bookings_silver')} DROP CONSTRAINT IF EXISTS {name}")
    spark.sql(f"ALTER TABLE {t('bookings_silver')} ADD CONSTRAINT {name} CHECK ({sql})")

silver_count = spark.table(t("bookings_silver")).count()
quarantine_count = spark.table(t("bookings_quarantine")).count()
print(f"Silver rows: {silver_count:,}   Quarantined rows: {quarantine_count:,}")

Silver rows: 118,587   Quarantined rows: 803


### Quality log
One row per rule per run: how many rows failed it.

In [0]:
rule_failures = (
    spark.table(t("bookings_quarantine"))
    .select(F.explode("_failed_rules").alias("rule"))
    .groupBy("rule")
    .count()
    .collect()
)
fail_by_rule = {r["rule"]: r["count"] for r in rule_failures}

log_rows = [
    (RUN_ID, name, int(fail_by_rule.get(name, 0)), bronze_count, fail_by_rule.get(name, 0) == 0)
    for name in RULES
]
log_df = spark.createDataFrame(
    log_rows, "run_id string, rule string, failed_rows long, total_rows long, passed boolean"
).withColumn("checked_at", F.current_timestamp())

log_df.write.format("delta").mode("append").saveAsTable(t("dq_check_log"))

# Fail loudly if quality drops past a threshold, instead of shipping bad data.
MAX_QUARANTINE_PCT = 1.0
quarantine_pct = 100 * quarantine_count / bronze_count
assert quarantine_pct <= MAX_QUARANTINE_PCT, (
    f"{quarantine_pct:.2f}% of rows failed quality checks (limit {MAX_QUARANTINE_PCT}%)"
)
assert silver_count + quarantine_count == bronze_count, "Rows were lost between bronze and silver"

log_df.orderBy("rule").show(truncate=False)

+----------------+---------------------------+-----------+----------+------+--------------------------+
|run_id          |rule                       |failed_rows|total_rows|passed|checked_at                |
+----------------+---------------------------+-----------+----------+------+--------------------------+
|20261026T223920Z|has_guests                 |180        |119390    |false |2026-10-26 22:40:06.992393|
|20261026T223920Z|has_stay_nights_or_canceled|622        |119390    |false |2026-10-26 22:40:06.992393|
|20261026T223920Z|non_negative_lead_time     |0          |119390    |true  |2026-10-26 22:40:06.992393|
|20261026T223920Z|non_negative_price         |1          |119390    |false |2026-10-26 22:40:06.992393|
|20261026T223920Z|valid_arrival_date         |0          |119390    |true  |2026-10-26 22:40:06.992393|
|20261026T223920Z|valid_cancel_flag          |0          |119390    |true  |2026-10-26 22:40:06.992393|
|20261026T223920Z|valid_hotel                |0          |119390

## Gold: cancellation metrics

In [0]:
gold = (
    spark.table(t("bookings_silver"))
    .groupBy("hotel", "arrival_date_year", "arrival_date_month", "market_segment")
    .agg(
        F.count("*").alias("bookings"),
        F.sum("is_canceled").alias("cancellations"),
        F.round(F.avg("is_canceled") * 100, 2).alias("cancel_rate_pct"),
        F.round(F.avg("adr"), 2).alias("avg_daily_rate"),
        F.round(F.avg("lead_time"), 1).alias("avg_lead_time_days"),
    )
)
gold.write.format("delta").mode("overwrite").option(
    "overwriteSchema", True
).saveAsTable(t("cancellation_metrics_gold"))

gold_count = spark.table(t("cancellation_metrics_gold")).count()
print(f"Gold rows: {gold_count:,}")

spark.sql(f"""
    SELECT hotel,
           SUM(bookings) AS bookings,
           ROUND(100 * SUM(cancellations) / SUM(bookings), 2) AS cancel_rate_pct
    FROM {t('cancellation_metrics_gold')}
    GROUP BY hotel
    ORDER BY cancel_rate_pct DESC
""").show()

Gold rows: 328
+------------+--------+---------------+
|       hotel|bookings|cancel_rate_pct|
+------------+--------+---------------+
|  City Hotel|   78912|          41.92|
|Resort Hotel|   39675|          28.03|
+------------+--------+---------------+



## Unity Catalog documentation and Delta history

In [0]:
docs = {
    "bookings_bronze": "Raw hotel bookings as loaded from CSV. All columns are strings.",
    "bookings_silver": "Cleaned, typed bookings that passed every data quality rule. "
                       "Leaky columns (reservation_status, reservation_status_date) removed.",
    "bookings_quarantine": "Rows that failed one or more quality rules; _failed_rules lists which.",
    "dq_check_log": "One row per quality rule per pipeline run.",
    "cancellation_metrics_gold": "Cancellation rate, average daily rate, and lead time by "
                                 "hotel, arrival month, and market segment.",
}
for table, comment in docs.items():
    spark.sql(f"COMMENT ON TABLE {t(table)} IS '{comment}'")

spark.sql(
    f"ALTER TABLE {t('cancellation_metrics_gold')} "
    "ALTER COLUMN cancel_rate_pct COMMENT 'Percent of bookings canceled'"
)

# Delta keeps every version of a table. Show silver's history.
spark.sql(f"DESCRIBE HISTORY {t('bookings_silver')}").select(
    "version", "operation"
).show(truncate=False)

+-------+---------------------------------+
|version|operation                        |
+-------+---------------------------------+
|3      |SET TBLPROPERTIES                |
|2      |ADD CONSTRAINT                   |
|1      |ADD CONSTRAINT                   |
|0      |CREATE OR REPLACE TABLE AS SELECT|
+-------+---------------------------------+



## Summary (numbers for the README)

In [0]:
summary = {
    "bronze_rows": bronze_count,
    "silver_rows": silver_count,
    "quarantined_rows": quarantine_count,
    "quality_rules": len(RULES),
    "rules_with_failures": sum(1 for n in RULES if fail_by_rule.get(n, 0)),
    "failures_by_rule": {k: v for k, v in sorted(fail_by_rule.items())},
    "columns_with_missing_values": cols_with_nulls,
    "leaky_columns_dropped": LEAKY,
    "gold_rows": gold_count,
    "delta_tables": len(docs),
}
for k, v in summary.items():
    print(f"{k}: {v}")

bronze_rows: 119390
silver_rows: 118587
quarantined_rows: 803
quality_rules: 7
rules_with_failures: 3
failures_by_rule: {'has_guests': 180, 'has_stay_nights_or_canceled': 622, 'non_negative_price': 1}
columns_with_missing_values: {'children': 4, 'country': 488, 'agent': 16340, 'company': 112593}
leaky_columns_dropped: ['reservation_status', 'reservation_status_date']
gold_rows: 328
delta_tables: 5
